# NeuroGolf-2026 — 5571.69 

**Public LB: 5571.69** (May 8 update — multi-source per-task best-pick under the 2026-05-06 scoring rules)

## Background — 2026-05-06 scoring change

On May 6 the official `neurogolf_utils.py` switched the objective formula from `25 - log(macs + memory + params)` to `25 - log(max(1, memory + params))`. **MACs no longer contribute**, memory is read from the ONNX Runtime profiler trace, and zero-cost networks now yield the full 25 points. Previously expensive Conv-heavy or fixed-shape rewrites that had been penalized for high MACs are now competitive again.

With MACs gone, many ONNX builds that had been retired (or never adopted) are suddenly the cheapest correct candidate for their task. The submission below is the result of evaluating a large pool of public + locally-built ONNX under the new formula and picking the lowest-cost correct one per task.

## What's new vs. the May 7 5211.69 baseline (+276.45 LB)

- **2026-04-29 broken-task rescue (10 tasks)**. Bisection identified that the post-May-5 grader rejects ONNX containing broadcast `Mul`/`Add`/`Sub` (e.g. `(1, 10, 30, 30) × (1, 1, 30, 1)`). All 10 broken tasks (t045/t067/t111/t159/t176/t192/t210/t256/t309/t320) were rescued by injecting `Tile` to align shapes before the broadcast op, plus an opset-13 / IR-8 / `BOOL`-output normalization pass.
- **Multi-source per-task swap under the new scoring**. 138 tasks now use a cheaper or correct ONNX than the previous baseline. Sources:
  - **afr_5377** (afr1ste's 5377.47 public artifact) — 119 tasks
  - **konbu17 local v62** — 14 tasks (post-MACs-removal, our prior `models/` rewrites become the cheapest correct option)
  - **sigmaborov** — 4 tasks (incl. one task where the previous solver was incorrect)
  - **svanikkolli** — 1 task
- **No remaining broken-task stubs.** Every task in this submission is a real ONNX that passes the post-May-6 strict shape inference + verification.

## What's inside

- **400 ONNX files** total
- All ONNX pass `is_strictly_compliant()` (4-D `input` / `output`, `dim_value > 0`, strict shape inference, opset 13 / IR 8 normalization)
- Banned ops (`LOOP`, `SCAN`, `NONZERO`, `UNIQUE`, `SCRIPT`, `FUNCTION`, `COMPRESS`) absent
- Total `submission.zip` size: **~1.13 MB**

## LB progression

| Date | Notes | LB |
|---|---|---:|
| May 4 | v77 (pre-grader-change) | 5264.22 |
| May 5 | New grader rejects 14 tasks → ERROR | (rejected) |
| May 5 | Stub workaround (14 tasks → all-False) | 5156.88 |
| May 5 | t211 / t371 alts recovered | 5182.96 |
| May 5 | t220 hand-built recovered | 5197.21 |
| May 6 | +15 task rewrites + 12 cheaper swaps | 5211.69 |
| May 7 | broadcast-`Mul` rescue: all 10 broken tasks restored | 5314.60 |
| May 7 | + blend-dataset task swap under new scoring | 5364.12 |
| May 7 | + afr1ste 5377.47 per-task swap | 5459.73 |
| May 7 | + multi-source best-pick (threshold 0.5) | 5474.55 |
| **May 8** | **+ multi-source best-pick (threshold 0.1, this notebook)** | **5571.69** |

## Compliance / scoring gate

Each ONNX in this submission was validated locally with the post-2026-05-06 official `neurogolf_utils.py`. Concretely:

1. `onnx.shape_inference.infer_shapes(model, strict_mode=True)` succeeds.
2. All `input` / `output` tensors are 4-D with `dim_value > 0`.
3. No banned ops.
4. Single input + single output, no functions / subgraphs, no name collisions.
5. Functional check on every `train + test + arc-gen` example.
6. `calculate_memory(model, ort_profile_trace)` and `calculate_params(model)` both yield non-negative values, and `points = max(1, 25 - log(max(1, memory + params)))` is computed per task to confirm the cheapest correct candidate is selected.

## Acknowledgements

Thanks to the original authors:

- **[@thisray](https://www.kaggle.com/thisray)** — gold-standard 395-task post-fix base.
- **[@jonathanchan](https://www.kaggle.com/jonathanchan)** — `ngc26-constraint-smart-logic-mix-blending` provided 31 compliant cheaper ONNX.
- **[@afr1ste](https://www.kaggle.com/afr1ste)** — `neurogolf-5192-83-current-rules-score` provided cheaper compliant ONNX (incl. 13 `GridSample`-based) for 73 tasks.
- **[@imaadmahmood](https://www.kaggle.com/imaadmahmood)**, **[@magmacot](https://www.kaggle.com/magmacot)**, **[@jazivxt](https://www.kaggle.com/jazivxt)** — earlier compliant ONNX contributions.

## Build `/kaggle/working/submission.zip`

The attached dataset ships the pre-built `submission.zip` (400 ONNX, deterministic). We copy it to `/kaggle/working/` and verify the per-ONNX manifest SHA-256 matches the pinned value.

In [ ]:
import hashlib, zipfile, os, re
from pathlib import Path

EXPECTED_FILE_COUNT = 400
EXPECTED_PUBLIC_SCORE = '5571.69'
EXPECTED_MANIFEST_SHA256 = 'a73756e5c47702db480dd14433179e7721a781dedeb2099a5556ac3b9104d0aa'

INPUT_ROOT = Path('/kaggle/input')
WORKING = Path('/kaggle/working'); WORKING.mkdir(exist_ok=True)
OUT = WORKING / 'submission.zip'

# Look for dataset-shipped submission.zip first (avoids re-zipping)
found_zips = list(INPUT_ROOT.rglob('submission.zip'))
if found_zips:
    src = found_zips[0]
    print(f'Using dataset-shipped {src}')
    import shutil; shutil.copy(src, OUT)
else:
    onnx_files = sorted(INPUT_ROOT.rglob('task*.onnx'))
    assert onnx_files, 'No task*.onnx or submission.zip found under /kaggle/input.'
    by_name = {}
    for f in onnx_files:
        if re.fullmatch(r'task\d{3}\.onnx', f.name) and f.name not in by_name:
            by_name[f.name] = f
    with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
        for name in sorted(by_name):
            z.write(by_name[name], arcname=name)

with zipfile.ZipFile(OUT) as z:
    names = sorted(n for n in z.namelist() if n.endswith('.onnx'))
    lines = [f'{n}\t{len(z.read(n))}\t{hashlib.sha256(z.read(n)).hexdigest()}' for n in names]
manifest_sha = hashlib.sha256('\n'.join(lines).encode()).hexdigest()

assert manifest_sha == EXPECTED_MANIFEST_SHA256, (manifest_sha, EXPECTED_MANIFEST_SHA256)
assert len(names) == EXPECTED_FILE_COUNT, len(names)

print(f'Wrote {OUT} for NeuroGolf public score {EXPECTED_PUBLIC_SCORE}')
print(f'  ONNX files:      {len(names)}')
print(f'  manifest sha256: {manifest_sha}')
print(f'  size:            {os.path.getsize(OUT):,} bytes')

## How to submit

Run all cells once, then click Kaggle's **Submit** button (top-right of this notebook in the Code editor) — it will use `/kaggle/working/submission.zip` automatically and does not consume a `kaggle competitions submit` quota until you press Submit.